In [ ]:
# Bengaluru House Price Prediction - End to End Linear Regression Project

# ==========================================
# 🧩 Step 1: Import Required Libraries
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 🧩 Step 2: Load and Inspect the Data
# ==========================================
data = pd.read_csv("bengaluru_house_prices.csv")
print("🔹 First 5 rows of the dataset:")
print(data.head())
print("\n🔹 Dataset Info:")
print(data.info())
print("\nDataset Shape:", data.shape)
print("\n🔹 Missing values per column:")
print(data.isnull().sum())

# ==========================================
# 🧩 Step 3: Drop Irrelevant Columns
# ==========================================
data = data.drop(['area_type', 'society', 'balcony', 'availability'], axis=1)
data = data.dropna()
print("\nAfter dropping missing values:", data.shape)

# ==========================================
# 🧩 Step 4: Feature Extraction — Extract BHK from Size
# ==========================================
data['BHK'] = data['size'].apply(lambda x: int(x.split(' ')[0]))

# ==========================================
# 🧩 Step 5: Convert 'total_sqft' to Numeric
# ==========================================
def convert_sqft_to_num(x):
    try:
        if '-' in x:
            tokens = x.split('-')
            return (float(tokens[0]) + float(tokens[1])) / 2
        if any(unit in x.lower() for unit in ['sq. meter', 'meter']):
            value = float(x.split(' ')[0])
            return value * 10.7639
        elif 'acre' in x.lower():
            value = float(x.split(' ')[0])
            return value * 43560
        elif 'yard' in x.lower():
            value = float(x.split(' ')[0])
            return value * 9
        elif 'cent' in x.lower():
            value = float(x.split(' ')[0])
            return value * 435.6
        elif 'perch' in x.lower():
            value = float(x.split(' ')[0])
            return value * 272.25
        elif 'guntha' in x.lower():
            value = float(x.split(' ')[0])
            return value * 1089
        elif 'ground' in x.lower():
            value = float(x.split(' ')[0])
            return value * 2400
        return float(x)
    except:
        return None

data['total_sqft'] = data['total_sqft'].astype(str).apply(convert_sqft_to_num)
data = data.dropna(subset=['total_sqft'])
print("\nAfter sqft conversion:", data.shape)

# ==========================================
# 🧩 Step 6: Feature Engineering — Price per Sqft
# ==========================================
data['price_per_sqft'] = data['price'] * 100000 / data['total_sqft']

# ==========================================
# 🧩 Step 7: Clean Location Data
# ==========================================
data['location'] = data['location'].apply(lambda x: x.strip())
location_stats = data['location'].value_counts(ascending=False)
rare_locs = location_stats[location_stats <= 10]
data['location'] = data['location'].apply(lambda x: 'other' if x in rare_locs else x)

# ==========================================
# 🧩 Step 8: Remove Outliers
# ==========================================
data = data[data['total_sqft'] / data['BHK'] >= 300]

def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

data = remove_pps_outliers(data)
print("\nAfter removing outliers:", data.shape)

# ==========================================
# 🧩 Step 9: Visualize Data Relationships
# ==========================================
plt.figure(figsize=(6,4))
sns.histplot(data['price_per_sqft'], bins=50, kde=True)
plt.title("Price per Sqft Distribution")
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='BHK', y='price', data=data)
plt.title("BHK vs Price")
plt.show()

# ==========================================
# 🧩 Step 10: Encoding Categorical Variable — Location
# ==========================================
dummies = pd.get_dummies(data['location'], drop_first=True)
data = pd.concat([data, dummies], axis=1)
data.drop('location', axis=1, inplace=True)

# ==========================================
# 🧩 Step 11: Prepare Feature Matrix and Target
# ==========================================
X = data.drop(['price', 'size'], axis=1)
y = data['price']

# ==========================================
# 🧩 Step 12: Train-Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 🧩 Step 13: Feature Scaling
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 🧩 Step 14: Train Linear Regression Model
# ==========================================
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# ==========================================
# 🧩 Step 15: Evaluate Model Performance
# ==========================================
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n🔹 Model Performance Metrics:")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.3f}")

plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Prices (Lakhs)")
plt.ylabel("Predicted Prices (Lakhs)")
plt.title("Actual vs Predicted House Prices")
plt.show()

print("\n✅ End-to-End Bengaluru House Price Prediction Model Completed Successfully!")
